<a href="https://colab.research.google.com/github/starsyntaxx/Olist-delivery-prediction/blob/main/notebooks/feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving final_olist_data.csv to final_olist_data.csv


In [3]:
import pandas as pd


In [4]:
df = pd.read_csv("final_olist_data.csv")

In [5]:
df.shape

(96455, 32)

In [6]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,freight_value_mean,product_name_lenght_mean,product_description_lenght_mean,product_photos_qty_mean,product_weight_g_sum,product_length_cm_mean,product_height_cm_mean,product_width_cm_mean,geolocation_lat_mean,geolocation_lng_mean
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,8.72,40.0,268.0,4.0,500.0,19.0,8.0,13.0,-23.680729,-46.444238
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,22.76,29.0,178.0,1.0,400.0,19.0,13.0,19.0,-19.807681,-43.980427
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,19.22,46.0,232.0,1.0,420.0,24.0,19.0,21.0,-21.363502,-48.229601
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,27.20,59.0,468.0,3.0,450.0,30.0,10.0,20.0,-19.837682,-43.924053
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,8.72,38.0,316.0,4.0,250.0,51.0,15.0,15.0,-23.543395,-46.262086


In [ ]:
df = df.drop(columns=[
    'order_delivered_carrier_date',
    'ge'
])

In [7]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [8]:
df['delivery_time_days'] = (
    df['order_delivered_customer_date'] -
    df['order_purchase_timestamp']
).dt.total_seconds() / 86400

In [9]:
df['approval_delay_days'] = (
    df['order_approved_at'] -
    df['order_purchase_timestamp']
).dt.total_seconds() / 86400

In [10]:
df['carrier_delay_days'] = (
    df['order_delivered_carrier_date'] -
    df['order_approved_at']
).dt.total_seconds() / 86400

In [11]:
df['transit_time_days'] = (
    df['order_delivered_customer_date'] -
    df['order_delivered_carrier_date']
).dt.total_seconds() / 86400

In [12]:
df['delivery_late_days'] = (
    df['order_delivered_customer_date'] -
    df['order_estimated_delivery_date']
).dt.total_seconds() / 86400

Product measurement values

In [13]:
df['product_volume_cm3'] = (
    df['product_length_cm_mean'] *
    df['product_height_cm_mean'] *
    df['product_width_cm_mean']
)

In [14]:
df['product_density'] = (
    df['product_weight_g_sum'] /
    df['product_volume_cm3']
)

In [15]:
import numpy as np

df['log_weight'] = np.log1p(df['product_weight_g_sum'])

In [16]:
df['size_complexity'] = (
    df['product_length_cm_mean'] +
    df['product_height_cm_mean'] +
    df['product_width_cm_mean']
)

In [17]:
df['log_volume'] = np.log1p(df['product_volume_cm3'])